**Implementing gold layer with delta tables with fact and dimension tables.**

In [0]:
CREATE OR REPLACE TABLE retail_lakehouse.gold.dim_customer (
    CustomerSK BIGINT GENERATED ALWAYS AS IDENTITY,
    CustomerID INT,
    CustomerName STRING,
    Email STRING,
    City STRING,
    Address STRING,
    StartDate DATE,
    EndDate DATE,
    IsActive BOOLEAN
)
USING DELTA;


--initial load
INSERT INTO retail_lakehouse.gold.dim_customer
(
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)
SELECT
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    CURRENT_DATE(),
    DATE '9999-12-31',
    TRUE
FROM retail_lakehouse.silver.customers;

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.gold.dim_product AS
SELECT
    monotonically_increasing_id() AS ProductSK,
    ProductID,
    ProductName,
    Category,
    UnitPrice
FROM retail_lakehouse.silver.products;

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.gold.dim_store AS
SELECT
    monotonically_increasing_id() AS StoreSK,
    StoreID,
    StoreName,
    Region
FROM retail_lakehouse.silver.stores;

In [0]:
CREATE OR REPLACE TABLE retail_lakehouse.gold.fact_sales AS
SELECT
    monotonically_increasing_id() AS SalesSK,
    s.TransactionID,
    c.CustomerSK,
    p.ProductSK,
    st.StoreSK,
    s.Quantity,
    s.Quantity * p.UnitPrice AS Amount,
    s.TxnDate
FROM retail_lakehouse.silver.sales s
JOIN retail_lakehouse.gold.dim_customer c
    ON s.CustomerID = c.CustomerID
    AND c.IsActive = TRUE
JOIN retail_lakehouse.gold.dim_product p
    ON s.ProductID = p.ProductID
JOIN retail_lakehouse.gold.dim_store st
    ON s.StoreID = st.StoreID;

select * from retail_lakehouse.gold.fact_sales limit 10;

In [0]:
SELECT COUNT(*) 
FROM retail_lakehouse.gold.fact_sales; 
SELECT *
FROM retail_lakehouse.gold.fact_sales
LIMIT 10;